### Cell 1: Initialization and Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

from kg_pipeline import KGPipeline
from df_helpers import ResolveCoreferences, ExtractConcepts, df2Graph, graph2Df, contextual_proximity

# --- config ---
INPUT_FILE = "../data_input/dataset/musique/musique_1000_test.json"
OUTPUT_DIR = "../data_output/dataset/musique/ds1000"

# Set the upper limit for the processing quantity (None indicates processing all)
MAX_CONTEXTS = 1000  
# Whether to force re-generation (True: Ignore cache and re-run; False: Prioritize loading from cache)
REGENERATE = False 

# Initialize Pipeline instance
pipeline = KGPipeline(output_dir=OUTPUT_DIR)

d:\Anaconda_envs\envs\andelie\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 1 -Chunking

In [ ]:
# Load the original JSON data and split it into Chunks
if REGENERATE:
    df_chunks = pipeline.load_and_split_data(
        json_path=INPUT_FILE, 
        max_contexts=MAX_CONTEXTS
    )
else:
    # Try to load the cache
    chunk_cache = Path(OUTPUT_DIR) / "chunk.csv"
    if chunk_cache.exists():
        df_chunks = pd.read_csv(chunk_cache, sep="|")
    else:
        # If the cache does not exist, force re-generation
        df_chunks = pipeline.load_and_split_data(json_path=INPUT_FILE, max_contexts=MAX_CONTEXTS)

print(f"Loaded {len(df_chunks)} chunks.")
print(df_chunks.head(2))

Loaded 19998 chunks.
   context_id  chunk_id                           title  \
0           0         0             Grant's First Stand   
1           0         1  List of show business families   

                                                text  
0  Grant's First Stand information: Grant's First...  
1  List of show business families information: Ac...  


## Step 2 - Coreference Resolution

In [ ]:
# Resolve coreferences for the Chunks
resolved_path = Path(OUTPUT_DIR) / "resolved_chunks.csv"

if REGENERATE or not resolved_path.exists():
    df_chunks = ResolveCoreferences(df_chunks,max_workers=30)
    df_chunks.to_csv(resolved_path, sep="|", index=False)
else:
    print("Loading resolved chunks from cache...")
    df_chunks = pd.read_csv(resolved_path, sep="|")

print(df_chunks[['text', 'resolved_text']].head(2))

Loading resolved chunks from cache...
                                                text  \
0  Grant's First Stand information: Grant's First...   
1  List of show business families information: Ac...   

                                       resolved_text  
0  Grant's First Stand information: Grant's First...  
1  List of show business families information: Ac...  


## Step 3 -chunks Embeddings 

In [ ]:
# REGENERATE = True
final_emb_filename = "chunks_with_embeddings.parquet"
temp_title_filename = "temp_title_embeddings.parquet" # 临时文件
final_path = Path(OUTPUT_DIR) / final_emb_filename
temp_path = Path(OUTPUT_DIR) / temp_title_filename

# 1. The calculation will be recalculated only when REGENERATE is set to True; otherwise, it will be loaded directly.
if REGENERATE or not final_path.exists():
    print("🚀 [Notebook] Start the double vector calculation (Content + Title)...")

    # --- Step 1: Calculate the content vector ---
    # This will generate the 'embedding' column
    df_content = pipeline.compute_embeddings(
        df=df_chunks, 
        text_col='text', 
        id_col='chunk_id', 
        file_name=final_emb_filename 
    )

    # --- Step 2: Calculate the title vector ---
    # In order not to overwrite the original file, we first save it to a temporary file.
    # Note: pipeline defaults to saving the column name as 'embedding'
    df_title = pipeline.compute_embeddings(
        df=df_chunks, 
        text_col='title', 
        id_col='chunk_id', 
        file_name=temp_title_filename 
    )

    # --- Step 3: Merge and Rename in Notebook ---
    print("🔄 [Notebook] 正在合并 Title 向量...")
    
    # Extract the chunk_id and embedding from df_title, and rename the embedding to title_embedding.
    # Note: Here we only take the required two columns to prevent other columns from being duplicated
    df_title_subset = df_title[['chunk_id', 'embedding']].copy()
    df_title_subset.rename(columns={'embedding': 'title_embedding'}, inplace=True)
    
    # Merge the title embeddings into the main dataset
    df_final = pd.merge(df_content, df_title_subset, on='chunk_id', how='left')
    
    # --- Step 4: Manually save the merged results ---
    # Must convert numpy array back to list to save as Parquet (this is what Pipeline does internally, so we do it manually)
    for col in ['embedding', 'title_embedding']:
        if col in df_final.columns:
            # Check if the first row is a numpy array and convert it
            if not df_final[col].empty and isinstance(df_final[col].iloc[0], np.ndarray):
                df_final[col] = df_final[col].apply(lambda x: x.tolist())

    df_final.to_parquet(final_path, index=False)
    
    # Clean up temporary files
    if temp_path.exists():
        os.remove(temp_path)
        
    print(f"✅ [Notebook] Double vector merging completed! Saved to {final_path}")
    df_chunks = df_final # Update the df_chunks currently stored in the memory

else:
    print(f"🔍 [Notebook] Loading existing cache: {final_path}")
    df_chunks = pd.read_parquet(final_path)

# --- Validate data ---
print("-" * 30)
first_row = df_chunks.iloc[0]
print(f"Data Loaded. Columns: {df_chunks.columns.tolist()}")
if 'embedding' in df_chunks.columns:
    print(f"Content Vector Dim: {len(first_row['embedding'])}")
if 'title_embedding' in df_chunks.columns:
    print(f"Title Vector Dim:   {len(first_row['title_embedding'])}")

🚀 [Notebook] 开始双重向量计算 (Content + Title)...
🔍 [Pipeline] 加载嵌入缓存: ..\data_output\dataset\musique\ds1000\chunks_with_embeddings.parquet
⏳ [Pipeline] 开始计算 19998 条数据的嵌入向量...


Computing Embeddings:  27%|██▋       | 83/313 [01:24<03:42,  1.04it/s]

## Step 4 -Concept Extraction

In [ ]:
# REGENERATE = False
# Extract entities from the text in parallel
concepts_path = Path(OUTPUT_DIR) / "extracted_concepts.csv"

if REGENERATE or not concepts_path.exists():
    df_concepts = ExtractConcepts(dataframe=df_chunks, max_workers=30)
    df_concepts.to_csv(concepts_path, sep="|", index=False)
else:
    df_concepts = pd.read_csv(concepts_path, sep="|")

print(f"Extracted {len(df_concepts)} concepts.")

Extracted 224351 concepts.


## Step 5 Entity Standardization

In [ ]:
# This step includes the most complex logic:
# 1. Aggregating entity synonyms (Rich Text)
# 2. Computing entity embeddings
# 3. Aligning using HAC + Genealogy Penalty
# 4. Generating standard entity names
# This step includes the most complex logic: aggregating synonyms -> computing embeddings -> HAC + Genealogy Penalty -> generating standard names
REGENERATE = False

std_path = Path(OUTPUT_DIR) / "dp_extracted_concepts.csv"
if REGENERATE or not std_path.exists():
    # # Execute the standardization process (the files will be saved internally)
    df_concepts_std = pipeline.standardize_entities(df_concepts)
else:
    print(f"🔍 Loading cached standardized entities: {std_path}")
    df_concepts_std = pd.read_csv(std_path, sep="|")

print("Standardization sample:")
print(df_concepts_std[['Entity', 'Standard_Entity', 'cluster_id']].head())

🔍 加载缓存的标准化实体: ..\data_output\dataset\musique\ds1000\dp_extracted_concepts.csv
Standardization sample:
          Entity Standard_Entity  cluster_id
0           2001            2001         168
1  First Session   First Session         191
2           1961            1961         178
3      Blue Note       Blue Note          82
4    Grant Green     Grant Green         121


## Step 6 - QA data extraction

In [ ]:
# Extracting questions and answers pairs from the original data
qa_path = Path(OUTPUT_DIR) / "qa.csv"


REGENERATE =True
if REGENERATE or not qa_path.exists():
    df_qa = pipeline.extract_qa_pairs(INPUT_FILE, max_contexts=MAX_CONTEXTS)
else:
    print(f"🔍 Loading cached QA data: {qa_path}")
    df_qa = pd.read_csv(qa_path, sep="|")
REGENERATE =False

print(df_qa.head(2))

🔍 加载缓存的 QA 数据: ..\data_output\dataset\musique\ds1000\qa.csv
                                            question            answer  \
0          Who is the spouse of the Green performer?  Miquette Giraudy   
1  Who founded the company that distributed the f...      Mike Medavoy   

   context_id  
0           0  
1           1  


### Step 7 - Relation Extraction

In [ ]:
# 1. Prepare the entity mapping dictionary {chunk: {original_name: standard_name}}
entity_map = pipeline.generate_entity_map_for_graph(df_concepts_std)

# REGENERATE = True

# 2. Perform relation extraction
graph_path = Path(OUTPUT_DIR) / "graph.csv"

if REGENERATE or not graph_path.exists():
    # Use LLM to extract relationships and pass them into entity_map to achieve guided extraction
    relations_list = df2Graph(df_chunks, entity_map,max_workers=30)
    df_graph = graph2Df(relations_list)
    df_graph.to_csv(graph_path, sep="|", index=False)
else:
    df_graph = pd.read_csv(graph_path, sep="|")

print(f"Extracted {len(df_graph)} relations.")
print(df_graph.head())

d:\Code\jupyter\knowledge_graph\musique\kg_pipeline.py:379: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return valid.groupby(['context_id', 'chunk_id']).apply(


Extracted 190108 relations.
   context_id               node_1       node_2  \
0           0        First Session         2001   
1           0        First Session    Blue Note   
2           0  Grant's First Stand         1961   
3           0  Grant's First Stand  Grant Green   
4           0  Grant's First Stand    Blue Note   

                                                edge  chunk_id  
0                First Session was released in 2001.         0  
1  Earlier recordings made by Grant for Blue Note...         0  
2          Grant's First Stand was released in 1961.         0  
3  Grant's First Stand is the debut album by Amer...         0  
4  Grant's First Stand was recorded and released ...         0  


### Step 8 - Contextual Proximity

In [ ]:
# Calculate additional graph edges based on Chunk co-occurrence
# REGENERATE = False

df_proximity = contextual_proximity(df_graph)
df_proximity.to_csv(Path(OUTPUT_DIR) / "contextual_proximity.csv", sep="|", index=False)

print(f"Generated {len(df_proximity)} proximity edges.")
print(df_proximity.head())

Generated 392716 proximity edges.
   context_id                                          node_1  \
0         641  "(Can't Live Without Your) Love and Affection"   
2         641  "(Can't Live Without Your) Love and Affection"   
4         641  "(Can't Live Without Your) Love and Affection"   
5         641  "(Can't Live Without Your) Love and Affection"   
16        973                              "10,000 to 15,000"   

                   node_2 chunk_id  count                  edge  
0        "After the Rain"    12836      8  contextual_proximity  
2           Gunnar Nelson    12836     10  contextual_proximity  
4          Matthew Nelson    12836     10  contextual_proximity  
5                  Nelson    12836     10  contextual_proximity  
16  Battle of Blood River    19476      7  contextual_proximity  


## Step 9 - Merge Entities & Compute Dual Embeddings

In [ ]:
# --- New Step: Merge and Enrich ---
# Aggregate entity information and calculate vectors (vec_entity and vec_desc)
merged_path = Path(OUTPUT_DIR) / "concepts_merged_with_vectors.parquet"

if REGENERATE and merged_path.exists():
    print(f"🗑️ [REGENERATE] Delete the old aggregated cache: {merged_path}")
    os.remove(merged_path)

# Call the Pipeline (internal logic: calculate and save if no file exists, otherwise load)
df_merged_vectors = pipeline.merge_and_embed_concepts(df_concepts=df_concepts_std)

print("Merged Data Sample:")
print(df_merged_vectors[['Standard_Entity', 'description', 'vec_entity']].head(2))

🚀 [Pipeline] 开始实体聚合...
🔄 [Helper] 正在聚合实体数据...
✅ [Helper] 聚合完成。原始行数: 224351 -> 聚合后行数: 192709
🛠️ [Pipeline] 构建增强实体文本...
🚀 [Pipeline] 计算 Enriched Entity Vectors...


Vec: Entity: 100%|██████████| 3012/3012 [59:15<00:00,  1.18s/it]  


🚀 [Pipeline] 计算 Description Vectors...


Vec: Desc: 100%|██████████| 3012/3012 [52:05<00:00,  1.04s/it] 


💾 [Pipeline] 保存聚合向量表到: ..\data_output\dataset\musique\ds1000\concepts_merged_with_vectors.parquet
Merged Data Sample:
  Standard_Entity                                        description  \
0            1961  The year Grant Green's debut album, Grant's Fi...   
1       Blue Note  The record label that released Grant Green's d...   

                                          vec_entity  
0  [0.01651139, 0.025822053, -0.026199149, -0.008...  
1  [-0.036208298, 0.026029117, -0.04892347, 0.013...  
